In [1]:
from mibici.pipeline import load_stage
import pandas as pd

data = load_stage("structure")

trips = data["viajes"].copy()
stations = data["estaciones"].copy()

2026-10-18 07:25:47.361 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: X:\Repositorios\Personal\Kaggle\notebooks\mibici\mibici_predictions


2026-10-18 07:25:57.165 | INFO     | mibici.dataset:load_raw:166 - Partición train: 2024-10-01 a 2026-02-22
2026-10-18 07:25:57.175 | INFO     | mibici.dataset:load_raw:170 - Viajes cargados: 6,311,111


### 1.- Missing Values

In [2]:
def summarize_missing(df):
    summary = df.isna().sum().to_frame("missing_count")
    summary["missing_pct"] = df.isna().mean().mul(100).round(2)
    return summary.sort_values("missing_count", ascending=False)

display(summarize_missing(trips))
display(summarize_missing(stations))

,missing_count,missing_pct
Genero,53941,0.85
Año_de_nacimiento,1717,0.03
Usuario_Id,0,0.00
Viaje_Id,0,0.00
Inicio_del_viaje,0,0.00
Fin_del_viaje,0,0.00
Origen_Id,0,0.00
Destino_Id,0,0.00


,missing_count,missing_pct
id,0,0.0
name,0,0.0
latitude,0,0.0
longitude,0,0.0
dpcapacity,0,0.0


Los faltantes se concentran en `Genero` (53,941; 0.85 %) y
`Año_de_nacimiento` (1,717; 0.03 %). Se conservan los registros, pues las
columnas necesarias para contar viajes por estación y periodo no presentan nulos.

El catálogo de estaciones no presenta valores nulos.

### 2.- Duplicates

In [3]:

print(f"Viajes: filas duplicadas: {trips.duplicated().sum():,}")
print(f"Viajes: IDs repetidos: {trips['Viaje_Id'].duplicated().sum():,}")

print(f"Estaciones: filas duplicadas: {stations.duplicated().sum():,}")
print(f"Estaciones: IDs repetidos: {stations['id'].duplicated().sum():,}")

Viajes: filas duplicadas: 0
Viajes: IDs repetidos: 0
Estaciones: filas duplicadas: 0
Estaciones: IDs repetidos: 0


In [4]:
duplicate_trips = (
    trips.loc[trips["Viaje_Id"].duplicated(keep=False)]
    .sort_values("Viaje_Id")
)

display(duplicate_trips.head(20))

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id


In [5]:
unique_rows = trips.drop_duplicates()

conflicting_ids = unique_rows.loc[
    unique_rows["Viaje_Id"].duplicated(keep=False),
    "Viaje_Id",
].nunique()

print(f"IDs con registros diferentes: {conflicting_ids:,}")

IDs con registros diferentes: 0


In [6]:
#Verificamos que todo este bien y eliminamos los duplicados

clean_trips = trips.drop_duplicates().reset_index(drop=True)

display(clean_trips.duplicated().sum())

trips = clean_trips

np.int64(0)

En train no hay filas duplicadas ni `Viaje_Id` repetidos.
`remove_duplicate_trips` se conserva en la etapa `quality_fixes`
como salvaguarda (H-co-005).

### 3. Domain Constraints

In [7]:
from mibici.data_preparation.quality_fixes import remove_duplicate_trips

trips = remove_duplicate_trips(data["viajes"].copy())

In [8]:
birth_year = trips["Año_de_nacimiento"]

print(f"Año mínimo: {birth_year.min()}")
print(f"Año máximo: {birth_year.max()}")

birth_year_counts = birth_year.value_counts().sort_index()

display(birth_year_counts.head(10))
display(birth_year_counts.tail(10))

Año mínimo: 199
Año máximo: 2023


Año_de_nacimiento
199      164
200        5
1920     160
1939     266
1941     440
1942    1438
1944     105
1945    1066
1946     158
1947      59
Name: count, dtype: Int64

Año_de_nacimiento
2002    215309
2003    155849
2004    136550
2005     88690
2006     69437
2007     30696
2008     11938
2009      1975
2010        69
2023       526
Name: count, dtype: Int64

In [9]:
approx_age = (
    trips["Inicio_del_viaje"].dt.year
    - trips["Año_de_nacimiento"]
)

display(approx_age.describe())

invalid_year = (
    trips["Año_de_nacimiento"].le(0)
    | approx_age.lt(0)
).fillna(False)

print(f"Años inválidos básicos: {invalid_year.sum():,}")

count    6309394.0
mean     34.584938
std      14.401611
min            1.0
25%           26.0
50%           32.0
75%           40.0
max         1827.0
dtype: Float64

Años inválidos básicos: 0


In [10]:
age_review = trips.assign(approx_age=approx_age)

display(
    age_review.loc[age_review["approx_age"] > 100]
    .groupby(["Año_de_nacimiento", "approx_age"])
    .agg(
        trip_count=("Viaje_Id", "size"),
        user_count=("Usuario_Id", "nunique"),
    )
)

trip_count  user_count
Año_de_nacimiento approx_age                        
199               1826               113           1
                  1827                51           1
200               1824                 5           1
1920              104                130           1
                  105                 30           1

In [11]:
# Años identificados en las tablas anteriores: ¿sus usuarios registran otro año?
suspect_years = [199, 200, 1920, 2023]

suspect_users = trips.loc[
    trips["Año_de_nacimiento"].isin(suspect_years),
    "Usuario_Id",
].unique()

display(
    trips.loc[trips["Usuario_Id"].isin(suspect_users)]
    .groupby(["Usuario_Id", "Año_de_nacimiento"], dropna=False)
    .size()
    .rename("trip_count")
)

Usuario_Id  Año_de_nacimiento
566415      1920                 160
2470505     2023                  73
2470510     2023                   1
2470513     2023                   4
2470516     2023                  99
2470521     2023                  95
2470524     2023                   5
2470527     2023                  37
2470532     2023                 134
2470535     2023                  16
2470540     2023                  62
3370005     200                    5
4622552     199                  164
Name: trip_count, dtype: int64

In [12]:
age_review = trips.assign(approx_age=approx_age)

display(
    age_review.loc[age_review["approx_age"] > 80]
    .groupby(["Año_de_nacimiento", "approx_age"])
    .agg(
        trip_count=("Viaje_Id", "size"),
        user_count=("Usuario_Id", "nunique"),
    )
)

trip_count  user_count
Año_de_nacimiento approx_age                        
199               1826               113           1
                  1827                51           1
200               1824                 5           1
1920              104                130           1
                  105                 30           1
1939              85                  22           1
                  86                 202           1
                  87                  42           1
1941              83                 117           1
                  84                 285           1
                  85                  38           1
1942              82                 284           3
                  83                1002           2
                  84                 152           2
1944              81                  41           3
                  82                   1           1
1945              81                 144           2

In [13]:
# Edades aproximadas bajas
display(
    age_review.loc[age_review["approx_age"] < 16]
    .groupby(["Año_de_nacimiento", "approx_age"])
    .agg(
        trip_count=("Viaje_Id", "size"),
        user_count=("Usuario_Id", "nunique"),
    )
)

trip_count  user_count
Año_de_nacimiento approx_age                        
2009              15                  11           1
2023              1                  128           8
                  2                  398          10

Años claramente inválidos: 199 (164 viajes) y 200 (5 viajes),
de un usuario cada uno.

Edades extremas:
- 1920: 160 viajes de un usuario, edad aproximada de 104 a 105 años.
- 2023: 526 viajes de 10 usuarios, edad aproximada de 1 a 2 años.
  Sus `Usuario_Id` son consecutivos (2470505 a 2470540).

Ninguno de estos usuarios registra otro año de nacimiento en train.

También se revisaron 2,330 viajes con edades aproximadas de 81 a 87 años.
Se conservan, porque la edad elevada, por sí sola, no demuestra un error.
Los límites para marcar años inválidos siguen pendientes (H-co-006).

### 4.- Continuous Variables

In [14]:
from mibici.data_preparation.quality_fixes import remove_duplicate_trips

# Base conocida: structure + correcciones aprobadas.
# replace_invalid_birth_years no se aplica: sus límites siguen en revisión (H-co-006).
trips = remove_duplicate_trips(data["viajes"].copy())
stations = data["estaciones"].copy()

In [15]:
display(
    stations[["latitude", "longitude", "dpcapacity"]].describe()
)

,latitude,longitude,dpcapacity
count,484.000000,484.000000,484.000000
mean,20.678520,-103.361871,16.882231
std,0.022032,0.031626,4.126942
min,20.636130,-103.433215,0.000000
25%,20.665077,-103.384164,15.000000
50%,20.674390,-103.362443,15.000000
75%,20.688609,-103.339320,19.000000
max,20.739523,-103.297112,39.000000


In [16]:
# revisando las estaciones con capacidad cero

zero_capacity_stations = stations.loc[
    stations["dpcapacity"].eq(0)
]

display(zero_capacity_stations)

,id,name,latitude,longitude,dpcapacity
94,99,ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo,20.685108,-103.333746,0
104,109,ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.,20.723221,-103.385583,0
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,20.686920,-103.334670,0
129,138,ex-(ZPN-044),20.697050,-103.363220,0


In [17]:
#revisamos viajes con origen o destino en estaciones con capacidad cero
zero_capacity_usage = zero_capacity_stations[
    ["id", "name", "dpcapacity"]
].copy()

zero_capacity_usage["departures"] = (
    zero_capacity_usage["id"]
    .map(trips["Origen_Id"].value_counts())
    .fillna(0)
    .astype("int64")
)

zero_capacity_usage["arrivals"] = (
    zero_capacity_usage["id"]
    .map(trips["Destino_Id"].value_counts())
    .fillna(0)
    .astype("int64")
)

display(zero_capacity_usage)

,id,name,dpcapacity,departures,arrivals
94,99,ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo,0,0,0
104,109,ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.,0,0,0
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,0,0,0
129,138,ex-(ZPN-044),0,0,0


In [18]:
# Estaciones del catálogo sin actividad en train
used_ids = set(trips["Origen_Id"]) | set(trips["Destino_Id"])
station_activity = stations.assign(
    used_in_train=stations["id"].isin(used_ids),
    ex_prefix=stations["name"].str.startswith("ex-"),
)

display(pd.crosstab(
    station_activity["ex_prefix"],
    station_activity["used_in_train"],
    margins=True,
))
print(f"Mayor id usado en train: {max(used_ids)}")
print(f"Mayor id del catálogo: {stations['id'].max()}")

used_in_train,False,True,All
ex_prefix,,,
False,100,370,470
True,13,1,14
All,113,371,484


Mayor id usado en train: 402
Mayor id del catálogo: 519


Las estaciones 99, 109, 116 y 138 tienen capacidad cero y prefijo `ex-`,
sin viajes en train.

El catálogo (septiembre de 2026) tiene 484 estaciones; 113 no registran
actividad en train, entre ellas 13 de las 14 con prefijo `ex-`.
Se conservan en el catálogo. Su inclusión en los modelos queda
pendiente de verificar su estado operativo (H-co-007).

### 5. Inconsistency

In [19]:
from mibici.data_preparation.quality_fixes import remove_duplicate_trips

# Base conocida: structure + correcciones aprobadas.
# replace_invalid_birth_years no se aplica: sus límites siguen en revisión (H-co-006).
trips = remove_duplicate_trips(data["viajes"].copy())
stations = data["estaciones"].copy()

In [20]:
unknown_origins = ~trips["Origen_Id"].isin(stations["id"])
unknown_destinations = ~trips["Destino_Id"].isin(stations["id"])

print(f"Viajes con origen fuera del catálogo: {unknown_origins.sum():,}")
print(f"Viajes con destino fuera del catálogo: {unknown_destinations.sum():,}")

Viajes con origen fuera del catálogo: 0
Viajes con destino fuera del catálogo: 0


In [21]:
unknown_station_counts = pd.concat(
    [
        trips.loc[unknown_origins, "Origen_Id"]
        .value_counts()
        .rename("departures"),
        
        trips.loc[unknown_destinations, "Destino_Id"]
        .value_counts()
        .rename("arrivals"),
    ],
    axis=1,
).fillna(0).astype("int64")

unknown_station_counts.index.name = "station_id"

display(unknown_station_counts)

,departures,arrivals
station_id,,


Todos los `Origen_Id` y `Destino_Id` de train existen en el catálogo:
0 viajes con origen desconocido y 0 con destino desconocido (H-co-011).

### 6. Temporal Relationships

In [22]:
from mibici.data_preparation.quality_fixes import remove_duplicate_trips

# Base conocida: structure + correcciones aprobadas.
# replace_invalid_birth_years no se aplica: sus límites siguen en revisión (H-co-006).
trips = remove_duplicate_trips(data["viajes"].copy())
stations = data["estaciones"].copy()

In [23]:
duration_minutes = (
    trips["Fin_del_viaje"] - trips["Inicio_del_viaje"]
).dt.total_seconds() / 60

print(f"Duraciones negativas: {duration_minutes.lt(0).sum():,}")
print(f"Duraciones de cero: {duration_minutes.eq(0).sum():,}")

display(
    duration_minutes.describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)

Duraciones negativas: 0
Duraciones de cero: 12,636


count    6.311111e+06
mean     1.115182e+01
std      1.019726e+01
min      0.000000e+00
1%       2.333333e-01
25%      5.900000e+00
50%      9.616667e+00
75%      1.500000e+01
95%      2.426667e+01
99%      3.278333e+01
max      1.150077e+04
dtype: float64

In [24]:
# Duración cero por mes
trip_month = trips["Inicio_del_viaje"].dt.to_period("M")

zero_by_month = pd.DataFrame({
    "trips": trip_month.value_counts().sort_index(),
    "zero_duration": trip_month[duration_minutes.eq(0)].value_counts(),
}).fillna(0).astype("int64")
zero_by_month["zero_pct"] = (
    zero_by_month["zero_duration"] / zero_by_month["trips"] * 100
).round(3)

display(zero_by_month)

,trips,zero_duration,zero_pct
Inicio_del_viaje,,,
2024-10,418690,7732,1.847
2024-11,391089,197,0.050
2024-12,340049,146,0.043
2025-01,380184,205,0.054
2025-02,372529,194,0.052
2025-03,405110,216,0.053
2025-04,366193,185,0.051
2025-05,395891,189,0.048
2025-06,356611,278,0.078


In [25]:
zero_duration_trips = trips.loc[duration_minutes.eq(0)]

display(zero_duration_trips.head(10))

same_station_count = (
    zero_duration_trips["Origen_Id"]
    .eq(zero_duration_trips["Destino_Id"])
    .sum()
)

print(f"Duración cero, misma estación: {same_station_count:,}")
print(
    f"Duración cero, distinta estación: "
    f"{len(zero_duration_trips) - same_station_count:,}"
)

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
173,35847497,730474,M,1993,2024-10-01 06:28:00,2024-10-01 06:28:00,303,303
228,35847565,3471144,M,2001,2024-10-01 06:49:00,2024-10-01 06:49:00,350,350
246,35847600,3503134,F,2005,2024-10-01 06:54:00,2024-10-01 06:54:00,56,56
321,35847697,447774,M,1983,2024-10-01 07:19:00,2024-10-01 07:19:00,259,259
531,35847950,2260424,F,1984,2024-10-01 08:06:00,2024-10-01 08:06:00,329,329
549,35847970,1032417,M,1991,2024-10-01 08:10:00,2024-10-01 08:10:00,260,260
580,35848005,16627,M,1976,2024-10-01 08:15:00,2024-10-01 08:15:00,303,303
585,35848010,16627,M,1976,2024-10-01 08:16:00,2024-10-01 08:16:00,303,303
697,35848138,4462032,M,1997,2024-10-01 08:39:00,2024-10-01 08:39:00,340,340
737,35848196,2324757,M,1996,2024-10-01 08:46:00,2024-10-01 08:46:00,212,212


Duración cero, misma estación: 9,221
Duración cero, distinta estación: 3,415


In [26]:
display(
    zero_duration_trips["Inicio_del_viaje"]
    .dt.date.value_counts()
    .head(10)
)

display(
    zero_duration_trips.groupby(["Origen_Id", "Destino_Id"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

Inicio_del_viaje
2024-10-30    324
2024-10-11    303
2024-10-13    299
2024-10-22    298
2024-10-04    293
2024-10-16    290
2024-10-08    284
2024-10-09    282
2024-10-23    277
2024-10-02    276
Name: count, dtype: int64

Origen_Id  Destino_Id
75         75            161
11         11            149
35         35            142
51         51            132
52         52            125
271        271           125
50         50            125
34         34            123
54         54             98
86         86             89
dtype: int64

In [27]:
longest_trips = trips.loc[
    duration_minutes.nlargest(10).index
].copy()

longest_trips["duration_minutes"] = duration_minutes.loc[
    longest_trips.index
]

display(
    longest_trips[
        [
            "Viaje_Id",
            "Inicio_del_viaje",
            "Fin_del_viaje",
            "Origen_Id",
            "Destino_Id",
            "duration_minutes",
        ]
    ]
)

,Viaje_Id,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,duration_minutes
3654274,40028076,2025-07-20 08:35:05,2025-07-28 08:15:51,48,72,11500.766667
617485,36547926,2024-11-15 18:02:13,2024-11-19 06:36:59,176,281,5074.766667
170929,36040792,2024-10-14 12:25:00,2024-10-16 17:07:00,11,208,3162.000000
3684096,40062369,2025-07-22 18:48:48,2025-07-24 02:13:59,142,234,1885.183333
1823689,37937287,2025-02-23 13:17:55,2025-02-24 18:52:48,169,396,1774.883333
66876,35924694,2024-10-06 11:39:00,2024-10-07 13:35:00,54,83,1556.000000
5155203,41752086,2025-11-15 13:55:43,2025-11-16 15:16:05,134,134,1520.366667
2689284,38928931,2025-05-02 15:30:11,2025-05-03 16:48:46,53,58,1518.583333
532908,36451011,2024-11-09 14:56:41,2024-11-10 15:35:31,303,201,1478.833333
5526637,42171191,2025-12-15 17:30:53,2025-12-16 17:39:36,87,87,1448.716667


- No se encontraron duraciones negativas.
- Se detectaron 12,636 viajes con duración cero: 9,221 en la misma
  estación y 3,415 entre estaciones distintas.
  - 7,732 son de octubre de 2024, cuyas fechas no incluyen segundos:
    cualquier viaje de menos de un minuto aparece con duración cero.
  - Febrero de 2026 tiene 1,519 (0.56 %), concentrados entre el 12 y
    el 21 de febrero y mayormente en la misma estación.
  - El resto de los meses está entre 0.04 % y 0.10 %.

  Su inclusión en el objetivo de demanda queda pendiente (H-co-008).
- La mediana de duración es de 9.62 minutos y el percentil 99
  es de 32.78 minutos.
- La duración máxima es de 11,500.77 minutos, aproximadamente 8 días.
  Se inspeccionaron los diez viajes más largos: la duración calculada
  coincide con sus fechas.
- Se conservan los viajes largos para el análisis de demanda.
  No se asume que toda su duración corresponda a pedaleo.
  Su uso en variables de duración requiere revisión (H-co-009).
- Los viajes que cruzan de día o mes no se consideran errores.

### 7. Constant / Near-Constant Variables

In [28]:
from mibici.data_preparation.quality_fixes import remove_duplicate_trips

# Base conocida: structure + correcciones aprobadas.
# replace_invalid_birth_years no se aplica: sus límites siguen en revisión (H-co-006).
trips = remove_duplicate_trips(data["viajes"].copy())
stations = data["estaciones"].copy()

In [29]:
def summarize_constants(df):
    return pd.DataFrame({
        "unique_count": df.nunique(dropna=False),
        "top_value_pct": [
            df[column]
            .value_counts(normalize=True, dropna=False)
            .max() * 100
            for column in df.columns
        ],
    }).round(2)
    
    
display(summarize_constants(trips))
display(summarize_constants(stations))

,unique_count,top_value_pct
Viaje_Id,6311111,0.00
Usuario_Id,36317,0.09
Genero,3,71.76
Año_de_nacimiento,75,5.17
Inicio_del_viaje,5269632,0.00
Fin_del_viaje,5283792,0.00
Origen_Id,371,1.92
Destino_Id,371,2.49


,unique_count,top_value_pct
id,484,0.21
name,484,0.21
latitude,480,0.41
longitude,481,0.41
dpcapacity,9,41.53


No se detectaron columnas constantes ni una concentración extrema
en un único valor (máximo: `Genero`, 71.76 %). No se aplican
transformaciones por este criterio.